# 📊 Notebook 01 — Exploratory Data Analysis (EDA)

**Goal:** Load, explore and visualise the two medical datasets used to build our RAG knowledge base:
- **PubMedQA** — 1,000 annotated biomedical Q&A pairs from PubMed abstracts
- **MedQuAD** — 1,000 clinical Q&A pairs from NIH health information portals

**Outputs:**
- Distribution plots for answer length, final decisions, question types
- Word cloud of medical terms
- Statistics table for the paper

In [ ]:
# Cell 1: Install dependencies (run once)
# %pip install -q datasets pandas matplotlib seaborn wordcloud plotly

In [ ]:
# Cell 2: Imports
import sys
sys.path.insert(0, '..')  # so we can import from src/

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud

from src.rag.loader import load_pubmedqa, load_medquad
from src.utils import get_logger

logger = get_logger('01_eda')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('✅ Imports OK')

In [ ]:
# Cell 3: Load PubMedQA (1,000 samples)
pubmed_docs = load_pubmedqa(limit=1000)
print(f'PubMedQA: {len(pubmed_docs)} documents loaded')

# Convert to DataFrame for analysis
pubmed_df = pd.DataFrame([
    {
        'text': d.page_content,
        'text_len': len(d.page_content),
        'source': d.metadata.get('source', 'PubMedQA'),
        'final_decision': d.metadata.get('final_decision', 'unknown'),
    }
    for d in pubmed_docs
])
pubmed_df.head(3)

In [ ]:
# Cell 4: Load MedQuAD (1,000 samples)
medquad_docs = load_medquad(limit=1000)
print(f'MedQuAD: {len(medquad_docs)} documents loaded')

medquad_df = pd.DataFrame([
    {
        'text': d.page_content,
        'text_len': len(d.page_content),
        'source': d.metadata.get('source', 'MedQuAD'),
        'question_type': d.metadata.get('question_type', 'general'),
    }
    for d in medquad_docs
])
medquad_df.head(3)

In [ ]:
# Cell 5: Dataset summary statistics
combined_df = pd.concat([pubmed_df[['text_len', 'source']], medquad_df[['text_len', 'source']]], ignore_index=True)

summary = combined_df.groupby('source')['text_len'].agg(['count', 'mean', 'median', 'min', 'max'])
summary.columns = ['Count', 'Mean Length', 'Median Length', 'Min', 'Max']
summary = summary.round(1)
print('=== Dataset Summary ===')
print(summary.to_string())

In [ ]:
# Cell 6: Plot — Text length distribution by dataset
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, title, color) in zip(axes, [
    (pubmed_df, 'PubMedQA Text Length Distribution', 'steelblue'),
    (medquad_df, 'MedQuAD Text Length Distribution', 'darkorange'),
]):
    sns.histplot(df['text_len'], bins=40, kde=True, ax=ax, color=color)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Characters per document')
    ax.set_ylabel('Count')
    ax.axvline(df['text_len'].median(), color='red', linestyle='--', label=f'Median: {df["text_len"].median():.0f}')
    ax.legend()

plt.tight_layout()
plt.savefig('../data/results/01_text_length_distribution.png', bbox_inches='tight')
plt.show()
print('💾 Saved: 01_text_length_distribution.png')

In [ ]:
# Cell 7: PubMedQA — Final decision distribution (yes / no / maybe)
fig, ax = plt.subplots(figsize=(7, 5))
counts = pubmed_df['final_decision'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#f39c12', '#95a5a6']
bars = ax.bar(counts.index, counts.values, color=colors[:len(counts)], edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f'{val}\n({100*val/len(pubmed_df):.1f}%)', ha='center', va='bottom', fontsize=11)

ax.set_title('PubMedQA — Final Decision Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Decision')
ax.set_ylabel('Number of samples')
ax.set_ylim(0, counts.max() * 1.2)
plt.tight_layout()
plt.savefig('../data/results/02_pubmedqa_final_decision.png', bbox_inches='tight')
plt.show()

In [ ]:
# Cell 8: MedQuAD — Question type distribution (top 10)
fig, ax = plt.subplots(figsize=(10, 5))
top_types = medquad_df['question_type'].value_counts().head(10)
sns.barplot(x=top_types.values, y=top_types.index, ax=ax, palette='Blues_r')
ax.set_title('MedQuAD — Top 10 Question Types', fontsize=14, fontweight='bold')
ax.set_xlabel('Count')
ax.set_ylabel('Question Type')
plt.tight_layout()
plt.savefig('../data/results/03_medquad_question_types.png', bbox_inches='tight')
plt.show()

In [ ]:
# Cell 9: Word cloud — most frequent medical terms
import re

all_text = ' '.join(
    (d.page_content for d in pubmed_docs + medquad_docs)
)
# Remove common stopwords manually (simple approach)
stopwords = set([
    'the', 'a', 'an', 'and', 'or', 'is', 'are', 'was', 'were', 'be',
    'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
    'would', 'could', 'should', 'may', 'might', 'this', 'that', 'these',
    'those', 'of', 'in', 'to', 'for', 'with', 'on', 'at', 'from', 'by',
    'which', 'who', 'what', 'how', 'when', 'where', 'not', 'no', 'if',
    'as', 'it', 'its', 'also', 'can', 'more', 'than', 'but', 'about',
    'Question', 'Answer', 'Conclusion',
])

wc = WordCloud(
    width=900, height=450,
    background_color='white',
    max_words=150,
    stopwords=stopwords,
    colormap='Blues',
).generate(all_text)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Medical Knowledge Base — Frequent Terms', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../data/results/04_wordcloud.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Saved: 04_wordcloud.png')

In [ ]:
# Cell 10: Combined dataset overview — pie chart
fig, ax = plt.subplots(figsize=(7, 7))
sizes = [len(pubmed_docs), len(medquad_docs)]
labels = [f'PubMedQA\n({sizes[0]:,} docs)', f'MedQuAD\n({sizes[1]:,} docs)']
colors = ['#3498db', '#e67e22']
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 12},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
for t in autotexts:
    t.set_fontsize(13)
    t.set_color('white')
    t.set_fontweight('bold')
ax.set_title(f'Knowledge Base Composition (Total: {sum(sizes):,} documents)',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../data/results/05_dataset_composition.png', bbox_inches='tight')
plt.show()
print('✅ EDA complete. Results saved to data/results/')